# Regression

Regression in machine learning is a supervised learning technique that predicts a numerical value. It is used to predict continuous numerical values based on a set of input features mapped to output labels. It allows us to understand how the change in one or more input factors can be measured and mapped to a change in the output that is affected by the former change in the input parameters -> learns the relationship b/w input and output variables.

Regression models are used for trend forecasting, risk analysis, price prediction, etc.

## Simple Linear Regression

Linear regression is a statistical model used to find the line of best fit. What this means is that it is used to model the relationship between a dependent variable ($y$) and one or more independent variables ($x$). The line of best fit refers to the process where there is actual data and the model's predicted data, and the model tries to find the line that is closest to both the actual and predicted data, thereby minimizing the area between them.

When there is one independent variable, it is simple linear regression, and when there are multiple, it is multiple linear regression.

Linear regression assumes a linear relationship between the input variable(s) $x$ and the output variable $y$.

The formula for simple linear regression is: $$y = \beta_0 + \beta_1x + \epsilon$$
Where:
- $y$ is the dependent variable (the target)
- $x$ is the independent variable (the feature)
- $\beta_0$ is the y-intercept (the value of $y$ when $x=0$)
- $\beta_1$ is the slope of the line (the change in $y$ for a unit change in $x$).
- $\epsilon$ is the error term (the residuals of the change in $y$ that cannot be explained by $x$).

In [1]:
import sklearn as sk
import numpy as np
import torch
import statsmodels as ss
import jax
import jaxlib
import plotly.express as px

In [2]:
# creating the independent variables
weight = 0.7
bias = 0.3

# creating the data
start = 0
stop = 1
step = 0.02

X = np.arange(start, stop, step)
y = weight*X + bias

In [3]:
X[:10], y[:10], len(X), len(y)

(array([0.  , 0.02, 0.04, 0.06, 0.08, 0.1 , 0.12, 0.14, 0.16, 0.18]),
 array([0.3  , 0.314, 0.328, 0.342, 0.356, 0.37 , 0.384, 0.398, 0.412,
        0.426]),
 50,
 50)

In [4]:
X.shape, y.shape

((50,), (50,))

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train[:10], y_train[:10]

len(X_train), len(X_test)

(40, 10)

In [6]:
train_size = int(len(X) * 0.8)

X_train, y_train = X[:train_size], y[:train_size]
X_test, y_test = X[train_size:], y[train_size:]

len(X_train), len(y_train), type(X_train)

(40, 40, numpy.ndarray)

In [7]:
def plot_predictions(X_train: np.ndarray = X_train, 
                     y_train: np.ndarray = y_train, 
                     X_test: np.ndarray = X_test, 
                     y_test: np.ndarray = y_test, 
                     predictions=None):
    fig = px.scatter(x=X_train, y=y_train, labels={"x": "X", "y": "y"}, width=600, height=400)
    fig.add_scatter(x=X_test, y=y_test, mode='markers', name="Testing data")

    if predictions is not None:
        fig.add_scatter(x=X_test, y=predictions, mode='markers', name="Predictions")
    
    fig.show()

In [8]:
plot_predictions()

In [9]:
class SimpleLinearRegression:
    def __init__(self, learning_rate=0.01, n_iters=100):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.losses = []

    def fit(self, X: np.ndarray, y: np.ndarray):
        n_samples = len(X)
        
        """we make the weights and bias start off as random values and 
        later we will update them using gradient descent"""
        self.weights = np.random.rand()
        self.bias = np.random.rand()

        for epoch in range(self.n_iters):
            # fit the model using the formula and get the y_prediction
            y_pred = self.weights * X + self.bias

            # compute the loss
            loss = np.mean((y_pred - y) ** 2)
            self.losses.append(loss)

            # computing the gradients
            dw = -(2 / n_samples) * np.sum(X * (y - y_pred)) # since bias is already there when calculating y_pred
            db = -(2 / n_samples) * np.sum(y - y_pred)

            # update the weights and bias
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

            if epoch % 10 == 0:
                print(f"Epoch: {epoch}, loss: {loss: .4f}, weight: {self.weights}, bias: {self.bias}")

    def predict(self, X: np.ndarray):
        # make sure to call fit() before predict()
        if self.weights is None or self.bias is None:
            raise ValueError("Weights and bias cannot be none. Run fit() first")
        y_pred = self.weights * X + self.bias
        return y_pred


In [10]:
model = SimpleLinearRegression(0.1, 100)
model.fit(X_train, y_train)

Epoch: 0, loss:  0.0856, weight: 0.5859372584734772, bias: 0.1212049719354379
Epoch: 10, loss:  0.0005, weight: 0.6623465702209271, bias: 0.2992764847817382
Epoch: 20, loss:  0.0000, weight: 0.670411901173397, bias: 0.31094323170784643
Epoch: 30, loss:  0.0000, weight: 0.6733615472949139, bias: 0.31080723082569617
Epoch: 40, loss:  0.0000, weight: 0.6757360902769617, bias: 0.30991301698913765
Epoch: 50, loss:  0.0000, weight: 0.6778786185024291, bias: 0.3090426352186458
Epoch: 60, loss:  0.0000, weight: 0.6798305030492781, bias: 0.3082451102400727
Epoch: 70, loss:  0.0000, weight: 0.6816100585534773, bias: 0.30751766913249373
Epoch: 80, loss:  0.0000, weight: 0.6832325963624118, bias: 0.3068543897421346
Epoch: 90, loss:  0.0000, weight: 0.6847119777216415, bias: 0.30624962978852893


In [11]:
y_preds = model.predict(X_test)

In [12]:
plot_predictions(predictions=y_preds)

In [13]:
residuals_y_pred = y_test - y_preds

fig = px.scatter(x=y_preds, y=residuals_y_pred, 
                 labels={"x": "Predicted values", 
                         "y": "Errors/residuals"},
                         title="Residual plot")
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

In [14]:
model_2 = SimpleLinearRegression(0.1, 200)
model_2.fit(X_train, y_train)
model_2_y_preds = model_2.predict(X_test)
plot_predictions(predictions=model_2_y_preds)

Epoch: 0, loss:  0.0276, weight: 0.27166295015096986, bias: 0.5723494172394813
Epoch: 10, loss:  0.0098, weight: 0.2808331925505274, bias: 0.4783047631664962
Epoch: 20, loss:  0.0081, weight: 0.3157698833024638, bias: 0.45756746175864893
Epoch: 30, loss:  0.0067, weight: 0.3495240636318369, bias: 0.44330749102477984
Epoch: 40, loss:  0.0056, weight: 0.3804359567532801, bias: 0.4306379445234955
Epoch: 50, loss:  0.0046, weight: 0.4086302167240731, bias: 0.41910998190279247
Epoch: 60, loss:  0.0039, weight: 0.4343376026646648, bias: 0.408600825095623
Epoch: 70, loss:  0.0032, weight: 0.4577768858628565, bias: 0.3990190084024476
Epoch: 80, loss:  0.0027, weight: 0.47914813403139334, bias: 0.39028260024234956
Epoch: 90, loss:  0.0022, weight: 0.49863380577878713, bias: 0.3823170024991562
Epoch: 100, loss:  0.0018, weight: 0.5164002645226722, bias: 0.3750542063020064
Epoch: 110, loss:  0.0015, weight: 0.5325991957216153, bias: 0.36843220371050517
Epoch: 120, loss:  0.0013, weight: 0.5473689

In [15]:
px.scatter(x=model_2_y_preds, y=(y_test - model_2_y_preds),
           labels={"x": "Predicted values", "y": "Errors"},
           title="Residual plot for model_2").add_hline(y=0, line_dash="dash",
                                                         line_color="red").show()

We can see that from the 2 models, the second one performed better owing to the number of training iterations. However, there are some assumptions that need to be true when using linear regression.

In [17]:
def check_homoscedasticity(model_preds: np.ndarray,
                           y_test: np.ndarray):
        residuals = y_test - model_preds

        figure = px.scatter(x=model_preds, y=residuals,
                            labels={"x": "Model predictions", "y": "Residuals/Errors"},
                            title="Residual plot")
        figure.add_hline(y=0, line_dash="dash", line_color="red")
        figure.show()